# Fibroblastic focus — the point of no return

The companion model to `ipf_simulation_colab.ipynb`. That notebook builds the
alveolus; this one isolates the lesion itself and asks a sharper question:

**once a focus has formed, when can it still resolve?**

Two layers run here. An agent model puts active, elongated cells on a substrate
that stiffens where they deposit collagen. A reduced equation for the same
system,

```
dE/dt = k_dep · f(E) · (1 − E/E_max) − k_deg · (E − E_healthy)
```

has up to three roots. When it has three, the middle one is an unstable
**separatrix**: below it the lesion resolves on its own, above it the lesion
sustains itself even after the insult is withdrawn. That threshold is the point
of no return, and it can be located exactly rather than found by trial.

Run the cells in order.

In [ ]:
#@title 1 · Setup { display-mode: "form" }
import os, subprocess, sys

REPO = "https://github.com/Danpc11/lung-nematic.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
ROOT = "/content/lung-nematic"

if os.path.isdir(ROOT):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, ROOT], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "imageio", "imageio-ffmpeg", "seaborn"], check=True)

for name in [n for n in list(sys.modules) if n.startswith("simulations")]:
    del sys.modules[name]
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from simulations.fibrofocus import (
    FocusConfig, FocusSimulation, run_and_record,
    fixed_points, stiffness_velocity, integrate_lesion,
    critical_value, scan_two_parameters, detect_defects,
)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

commit = subprocess.run(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"repository ready at commit {commit}")

## Parameters

In [ ]:
#@title 2 · Cells and mechanics { display-mode: "form" }

#@markdown Domain, duration and how long the epithelial insult lasts.
days_to_simulate = 14.0  #@param {type:"slider", min:2, max:60, step:1}
insult_duration_days = 4.0  #@param {type:"slider", min:0, max:20, step:0.5}
domain_um = 600.0  #@param {type:"slider", min:300, max:1200, step:50}
n_initial_cells = 220  #@param {type:"slider", min:50, max:600, step:10}

#@markdown Cell shape sets both the excluded volume and the smallest resolvable
#@markdown feature of the director field.
rod_length_um = 50.0  #@param {type:"slider", min:20, max:90, step:5}
rod_width_um = 11.0  #@param {type:"slider", min:5, max:25, step:1}

#@markdown Activity: speed, orientational noise and alignment set the density of
#@markdown topological defects through the active length.
speed_um_per_h = 21.4  #@param {type:"number"}
rot_diffusion_per_h = 0.80  #@param {type:"number"}
align_rate_per_h = 1.4  #@param {type:"number"}
durotaxis_um2_per_kPa_h = 55.0  #@param {type:"number"}
prolif_rate_per_h = 0.022  #@param {type:"number"}

mechanics = dict(
    total_time_h=days_to_simulate * 24.0,
    injury_duration_h=insult_duration_days * 24.0,
    width_um=domain_um, height_um=domain_um,
    n_initial=int(n_initial_cells),
    rod_length_um=rod_length_um, rod_width_um=rod_width_um,
    speed_um_per_h=speed_um_per_h,
    rot_diffusion_per_h=rot_diffusion_per_h,
    align_rate_per_h=align_rate_per_h,
    durotaxis_um2_per_kPa_h=durotaxis_um2_per_kPa_h,
    prolif_rate_per_h=prolif_rate_per_h,
)

# Same counting-noise arithmetic as the alveolar notebook: a window needs about
# 30 orientations before measured order can be told apart from chance.
area = np.pi * 0.25 * rod_length_um * rod_width_um
print(f"cell footprint {area:.0f} um^2 -> director windows need radius "
      f">= {np.sqrt(30*area/np.pi):.0f} um to beat counting noise")

In [ ]:
#@title 3 · Matrix, activation and the insult { display-mode: "form" }

#@markdown **The competition that sets the separatrix.** Deposition against
#@markdown turnover: their ratio decides whether a lesion can resolve.
deposition_rate_kPa_per_h = 0.10  #@param {type:"number"}
degradation_rate_per_h = 0.004  #@param {type:"number"}

#@markdown **Activation and hysteresis.** Cells switch on above `E_act` but only
#@markdown switch off below `E_act / memory_factor` — that gap is mechanical
#@markdown memory, and it is what makes the system bistable rather than merely
#@markdown threshold-driven.
E_healthy_kPa = 2.0  #@param {type:"slider", min:0.2, max:6, step:0.1}
E_act_kPa = 16.0  #@param {type:"slider", min:4, max:40, step:0.5}
E_max_kPa = 100.0  #@param {type:"slider", min:30, max:200, step:5}
memory_factor = 3.0  #@param {type:"slider", min:1, max:10, step:0.1}
activation_rate_per_h = 0.08  #@param {type:"number"}
deactivation_rate_per_h = 0.004  #@param {type:"number"}

#@markdown **The insult.** A transient epithelial injury that lowers the
#@markdown activation threshold and lays down provisional matrix.
injury_radius_um = 70.0  #@param {type:"slider", min:20, max:200, step:5}
injury_provisional_E_kPa = 9.0  #@param {type:"slider", min:2, max:30, step:0.5}
injury_activation_drop_kPa = 12.0  #@param {type:"number"}

random_seed = 0  #@param {type:"integer"}

matrix = dict(
    deposition_rate_kPa_per_h=deposition_rate_kPa_per_h,
    degradation_rate_per_h=degradation_rate_per_h,
    E_healthy_kPa=E_healthy_kPa, E_act_kPa=E_act_kPa, E_max_kPa=E_max_kPa,
    memory_factor=memory_factor,
    activation_rate_per_h=activation_rate_per_h,
    deactivation_rate_per_h=deactivation_rate_per_h,
    injury_radius_um=injury_radius_um,
    injury_provisional_E_kPa=injury_provisional_E_kPa,
    injury_activation_drop_kPa=injury_activation_drop_kPa,
    seed=int(random_seed),
)

from dataclasses import replace
config = replace(FocusConfig(), **mechanics, **matrix)
config.validate()
print(f"deactivation threshold sits at E_act / memory = "
      f"{E_act_kPa/memory_factor:.1f} kPa")
print(f"provisional matrix after injury: {injury_provisional_E_kPa:.1f} kPa")

In [ ]:
#@title 4 · The point of no return { display-mode: "form" }
#@markdown Plots dE/dt against E. Every zero crossing is a steady state: filled
#@markdown circles are stable, the open circle is the **separatrix**. Shading
#@markdown marks the basin that resolves and the basin that does not.
figure_width = "double column (183 mm)"  #@param ["single column (89 mm)", "double column (183 mm)"]
show_trajectory = True  #@param {type:"boolean"}

import matplotlib as mpl
PALETTE = {"blue":"#0072B2","vermillion":"#D55E00","green":"#009E73",
           "orange":"#E69F00","sky":"#56B4E9","pink":"#CC79A7","grey":"#999999"}
MM = 1/25.4
width_in = (89.0 if figure_width.startswith("single") else 183.0) * MM

sns.set_theme(style="ticks", context="paper")
mpl.rcParams.update({
    "font.family":"sans-serif","font.sans-serif":["DejaVu Sans","Arial"],
    "font.size":7,"axes.labelsize":7,"xtick.labelsize":6,"ytick.labelsize":6,
    "legend.fontsize":6,"legend.frameon":False,"axes.linewidth":0.5,
    "xtick.major.width":0.5,"ytick.major.width":0.5,
    "xtick.major.size":2.5,"ytick.major.size":2.5,"lines.linewidth":1.1,
    "savefig.bbox":"tight","savefig.pad_inches":0.02,
    "pdf.fonttype":42,"ps.fonttype":42,
})

roots = fixed_points(config)
E_axis = np.linspace(config.E_healthy_kPa, config.E_max_kPa, 1200)
velocity = stiffness_velocity(E_axis, config)

n_panels = 2 if show_trajectory else 1
figure, axes = plt.subplots(1, n_panels,
                            figsize=(width_in, width_in * 0.34 * (1 if n_panels==2 else 2)))
axes = np.atleast_1d(axes)

ax = axes[0]
ax.axhline(0.0, color="0.7", lw=0.5, zorder=0)
if roots["bistable"]:
    sep = roots["E_separatrix"]
    ax.axvspan(config.E_healthy_kPa, sep, color=PALETTE["green"], alpha=0.10, lw=0)
    ax.axvspan(sep, config.E_max_kPa, color=PALETTE["vermillion"], alpha=0.10, lw=0)
ax.plot(E_axis, velocity, color=PALETTE["blue"])
for value in roots["stable"]:
    ax.plot(value, 0.0, "o", ms=4, mfc=PALETTE["blue"], mec=PALETTE["blue"], zorder=5)
for value in roots["unstable"]:
    ax.plot(value, 0.0, "o", ms=4, mfc="white", mec=PALETTE["vermillion"],
            mew=1.0, zorder=5)
ax.set_xlabel("matrix stiffness E (kPa)")
ax.set_ylabel("dE/dt (kPa h$^{-1}$)")
sns.despine(ax=ax)
ax.text(-0.16, 1.04, "a", transform=ax.transAxes, fontsize=8, fontweight="bold")

if show_trajectory:
    result = integrate_lesion(config)
    trace = np.asarray(result["trace"])          # 1-D, one entry per timestep
    hours = np.arange(len(trace)) * config.dt_h
    ax = axes[1]
    ax.plot(hours / 24.0, trace, color=PALETTE["vermillion"])
    if roots["bistable"]:
        ax.axhline(roots["E_separatrix"], color=PALETTE["grey"], lw=0.6,
                   ls=(0,(3,1.5)), label="separatrix")
        ax.legend(handlelength=1.2, borderpad=0.2)
    ax.axvline(config.injury_duration_h/24.0, color="0.6", lw=0.5, ls=(0,(2,2)))
    ax.set_xlabel("time (days)"); ax.set_ylabel("E (kPa)")
    sns.despine(ax=ax)
    ax.text(-0.16, 1.04, "b", transform=ax.transAxes, fontsize=8, fontweight="bold")

figure.tight_layout(pad=0.4, w_pad=1.8)
plt.show()

print(f"bistable: {roots['bistable']}")
if roots["bistable"]:
    print(f"  healthy attractor   E = {roots['E_healthy']:6.2f} kPa")
    print(f"  SEPARATRIX          E = {roots['E_separatrix']:6.2f} kPa   <- the point of no return")
    print(f"  fibrotic attractor  E = {roots['E_fibrotic']:6.2f} kPa")
    print()
    print(f"  the insult leaves provisional matrix at {config.injury_provisional_E_kPa:.1f} kPa, which is")
    print(f"  {'ABOVE' if config.injury_provisional_E_kPa > roots['E_separatrix'] else 'below'}"
          f" the separatrix -> the lesion "
          f"{'cannot' if config.injury_provisional_E_kPa > roots['E_separatrix'] else 'can'} resolve on its own")
else:
    print("  only one attractor: the outcome is set by the insult, not by a threshold")

In [ ]:
#@title 5 · Where exactly is the threshold? { display-mode: "form" }
#@markdown Bisects for the critical value of one parameter — the value at which
#@markdown the lesion flips between resolving and persisting. This is the number
#@markdown a therapeutic target would have to move.
parameter = "deposition_rate_kPa_per_h"  #@param ["deposition_rate_kPa_per_h", "degradation_rate_per_h", "memory_factor", "injury_duration_h", "injury_provisional_E_kPa", "E_act_kPa"]
search_low = 0.01  #@param {type:"number"}
search_high = 0.60  #@param {type:"number"}

value = critical_value(config, parameter, search_low, search_high)
current = getattr(config, parameter)

if np.isnan(value):
    print(f"No threshold between {search_low} and {search_high}:")
    print("both ends give the same outcome. Widen the bracket.")
else:
    print(f"critical {parameter} = {value:.4f}")
    print(f"currently set to        {current:.4f}")
    margin = (current - value) / value * 100
    side = "past" if current > value else "short of"
    print(f"the lesion sits {abs(margin):.0f} % {side} the threshold")
    print()
    print("Read this as the distance a drug would need to close, not as a dose.")

In [ ]:
#@title 6 · Phase diagram over two parameters { display-mode: "form" }
#@markdown Integrates the reduced model across a grid and maps where lesions
#@markdown persist. The boundary is the separatrix projected onto two axes.
x_parameter = "deposition_rate_kPa_per_h"  #@param ["deposition_rate_kPa_per_h", "degradation_rate_per_h", "memory_factor", "injury_provisional_E_kPa", "E_act_kPa"]
x_min = 0.02  #@param {type:"number"}
x_max = 0.30  #@param {type:"number"}
y_parameter = "degradation_rate_per_h"  #@param ["degradation_rate_per_h", "deposition_rate_kPa_per_h", "memory_factor", "injury_provisional_E_kPa", "E_act_kPa"]
y_min = 0.001  #@param {type:"number"}
y_max = 0.020  #@param {type:"number"}
grid_points = 16  #@param {type:"slider", min:6, max:32, step:2}

if x_parameter == y_parameter:
    raise ValueError("choose two different parameters")

x_values = np.linspace(x_min, x_max, int(grid_points))
y_values = np.linspace(y_min, y_max, int(grid_points))
scan = scan_two_parameters(config, x_parameter, x_values, y_parameter, y_values)

grid = scan.pivot_table(index=y_parameter, columns=x_parameter, values="E_final_kPa")
persist = scan.pivot_table(index=y_parameter, columns=x_parameter, values="persisted")

figure, ax = plt.subplots(figsize=(width_in * 0.62, width_in * 0.46))
mesh = ax.pcolormesh(grid.columns.values, grid.index.values, grid.values,
                     cmap="rocket_r", shading="auto", rasterized=True)
bar = figure.colorbar(mesh, ax=ax, pad=0.02)
bar.set_label("final stiffness (kPa)")
bar.outline.set_linewidth(0.5)
ax.contour(persist.columns.values, persist.index.values,
           persist.values.astype(float), levels=[0.5],
           colors="white", linewidths=1.0, linestyles="--")
ax.set_xlabel(x_parameter.replace("_", " "))
ax.set_ylabel(y_parameter.replace("_", " "))
ax.plot(getattr(config, x_parameter), getattr(config, y_parameter),
        marker="*", ms=9, color="white", mec="black", mew=0.5)
sns.despine(ax=ax, top=False, right=False)
figure.tight_layout(pad=0.4)
plt.show()

print(f"dashed line: the persistence boundary   |   star: current settings")
print(f"lesions persist in {scan['persisted'].mean()*100:.0f} % of the grid")

In [ ]:
#@title 7 · Run the agent model { display-mode: "form" }
#@markdown The reduced equation locates thresholds; this runs the spatial model
#@markdown it was reduced *from*, so the two can be compared.

frames_per_run = 60  #@param {type:"slider", min:10, max:150, step:5}
frames_per_second = 12  #@param {type:"slider", min:2, max:24, step:1}

#@markdown Overlay the detected defects on the video. Note that the built-in
#@markdown overlay uses a fixed narrow window; cell 8 re-tests those detections
#@markdown properly and usually rejects them.
show_defects_overlay = True  #@param {type:"boolean"}
export_mp4 = True  #@param {type:"boolean"}
export_gif = True  #@param {type:"boolean"}

import shutil, os
# Fresh directory per run: a previous run's MP4/GIF must not leak into the ZIP
# if an export is toggled off this time.
results_dir = "/content/focus_results"
if os.path.isdir(results_dir):
    shutil.rmtree(results_dir)
os.makedirs(results_dir, exist_ok=True)

import time
frame_every_h = config.total_time_h / frames_per_run
start = time.time()
outputs = run_and_record(
    config, "/content/focus_results", frame_every_h=frame_every_h,
    fps=frames_per_second, show_defects=show_defects_overlay,
    make_mp4=export_mp4, make_gif=export_gif,
)
elapsed = time.time() - start
final = outputs.get("final", {})
print(f"finished in {elapsed:.0f} s, {outputs['n_frames']} frames\n")
for key in ("n_cells", "myo_fraction", "E_focus_kPa", "fibrotic_area_frac",
            "global_order_S", "density_per_um2"):
    if key in final:
        print(f"  {key:22s} {final[key]:.4g}")

# the trajectory the reduced model predicted, for comparison
predicted = integrate_lesion(config)
print()
print(f"  reduced model predicted E_final = {predicted['E_final_kPa']:.1f} kPa "
      f"({'persists' if predicted['persisted'] else 'resolves'})")
if "E_focus_kPa" in final:
    print(f"  agent model reached             = {final['E_focus_kPa']:.1f} kPa")

In [ ]:
#@title 8 · Watch it { display-mode: "form" }
from base64 import b64encode
from IPython.display import HTML, Image, display

if export_mp4 and "mp4" in outputs:
    data = b64encode(open(outputs["mp4"], "rb").read()).decode()
    display(HTML(f'''<video width="820" controls loop>
        <source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'''))
elif export_gif and "gif" in outputs:
    display(Image(filename=outputs["gif"]))
else:
    print("Enable a video export in cell 7 to preview.")

In [ ]:
#@title 9 · Defects, tested against counting noise { display-mode: "form" }
#@markdown A director field built from discrete cells carries counting noise:
#@markdown N randomly oriented rods produce apparent order of order 1/sqrt(N) by
#@markdown chance alone. This cell sizes the window from the measured density and
#@markdown then tests every window against the null **for its own sample count**,
#@markdown rather than against one global threshold.

min_samples_per_window = 30  #@param {type:"slider", min:5, max:100, step:5}
significance_quantile = 0.95  #@param {type:"slider", min:0.5, max:0.999, step:0.005}
window_choice = "adaptive (from measured density)"  #@param ["adaptive (from measured density)", "fixed"]
fixed_window_um = 25.0  #@param {type:"slider", min:5, max:150, step:5}
compare_with_naive = True  #@param {type:"boolean"}

from simulations.nematic_resolution import resolved_defects, null_order_quantile

cell_area = np.pi * 0.25 * config.rod_length_um * config.rod_width_um
# Reuse the simulation returned by cell 7 instead of running it all again.
sim_final = outputs["simulation"]

sigma = None if window_choice.startswith("adaptive") else float(fixed_window_um)
result = resolved_defects(
    sim_final.x, sim_final.y, sim_final.theta,
    (config.width_um, config.height_um), cell_area,
    grid_step_um=config.grid_step_um,
    min_samples=int(min_samples_per_window),
    quantile=float(significance_quantile),
    sigma_um=sigma,
)
info = result["diagnostics"]

print("WINDOW")
print(f"  window used                     {info['sigma_um']:6.1f} um")
print(f"  needed at full packing (R_min)  {info['r_min_at_full_packing_um']:6.1f} um"
      f"   = sqrt(N * A_cell / pi)")
print(f"  adequate                        {info['window_is_adequate']}")
print(f"  -> defect pairs closer than     {info['resolution_limit_um']:6.1f} um cannot be separated")
print()
print("ORDER")
print(f"  samples per window (typical)    {info['typical_samples_per_window']:6.1f}")
print(f"  order expected by chance        {info['null_quantile_at_typical_n']:6.3f}")
print(f"  order measured where significant{info['median_order_where_significant']:6.3f}")
print(f"  area with real nematic order    {info['significant_area_fraction']*100:5.1f} %")
print()
print("DEFECTS")
print(f"  surviving the null test         +{len(result['plus'])} / -{len(result['minus'])}")

if compare_with_naive:
    naive = detect_defects(sim_final, sigma_um=float(fixed_window_um))
    n_naive = len(naive["plus"]) + len(naive["minus"])
    n_kept = len(result["plus"]) + len(result["minus"])
    print()
    print(f"  a fixed {fixed_window_um:.0f} um window without the test reports {n_naive}")
    if n_naive and not n_kept:
        print("  -> all of them are counting noise")
    elif n_naive:
        print(f"  -> {n_naive - n_kept} of {n_naive} were noise")

print()
if info["significant_area_fraction"] > 0.05 and not (len(result["plus"]) + len(result["minus"])):
    print("Real nematic order is present, but no defect survives at the resolution")
    print("the cell size permits. Order and defects are separate claims: the first")
    print("can be supported here, the second cannot.")

In [ ]:
#@title 10 · Where the order is real { display-mode: "form" }
#@markdown Left: nematic order. Right: only the windows that beat their own null.
#@markdown Everything masked out is indistinguishable from randomly oriented cells.
save_figure = True  #@param {type:"boolean"}

field = result["field"]
significant = result["significant"]
extent = [0, config.width_um, 0, config.height_um]

figure, axes = plt.subplots(1, 2, figsize=(width_in, width_in * 0.42))

image = axes[0].imshow(field["order"], origin="lower", extent=extent,
                       cmap="mako", vmin=0, vmax=1, rasterized=True)
bar = figure.colorbar(image, ax=axes[0], fraction=0.046, pad=0.02)
bar.set_label("order S"); bar.outline.set_linewidth(0.5)
axes[0].set_title("measured", fontsize=7)

masked = np.where(significant, field["order"], np.nan)
image = axes[1].imshow(masked, origin="lower", extent=extent,
                       cmap="mako", vmin=0, vmax=1, rasterized=True)
bar = figure.colorbar(image, ax=axes[1], fraction=0.046, pad=0.02)
bar.set_label("order S"); bar.outline.set_linewidth(0.5)
axes[1].set_title(f"above its own null (p < {1-significance_quantile:.2f})", fontsize=7)

for defects, marker, colour in ((result["plus"], "+", "white"),
                                (result["minus"], "x", "#FFD166")):
    if len(defects):
        axes[1].scatter(defects[:, 0], defects[:, 1], marker=marker, s=40,
                        linewidths=1.0, color=colour)

for letter, ax in zip("ab", axes):
    ax.set_xlabel("x (µm)"); ax.set_aspect("equal")
    ax.text(-0.12, 1.04, letter, transform=ax.transAxes,
            fontsize=8, fontweight="bold")
axes[0].set_ylabel("y (µm)")
figure.tight_layout(pad=0.4, w_pad=1.4)
if save_figure:
    from pathlib import Path
    Path("/content/focus_results").mkdir(parents=True, exist_ok=True)
    figure.savefig("/content/focus_results/order_significance.pdf")
    figure.savefig("/content/focus_results/order_significance.png", dpi=600)
plt.show()

In [ ]:
#@title 11 · Export { display-mode: "form" }
import json, zipfile
from dataclasses import asdict
from pathlib import Path

bundle_name = "fibrofocus_run"  #@param {type:"string"}

results = Path("/content/focus_results")
results.mkdir(parents=True, exist_ok=True)
with open(results / "config.json", "w") as handle:
    json.dump(asdict(config), handle, indent=2)
scan.to_csv(results / "phase_scan.csv", index=False)

bundle = Path(f"/content/{bundle_name}.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for item in results.iterdir():
        if item.is_file():
            archive.write(item, item.name)

print(f"{bundle.name}  ({bundle.stat().st_size/1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(str(bundle))
except Exception as error:
    print(f"Automatic download unavailable ({error}); use the file browser.")

## How to read this

**The separatrix is the result, its value is not.** That three roots exist, and
that the middle one separates a lesion that resolves from one that does not, is
a structural property of deposition competing with turnover. The particular
number it lands on depends on rates that are order-of-magnitude estimates. Quote
the structure; treat the number as provisional.

**Critical values are distances, not doses.** Cell 5 reports how far a parameter
sits from its threshold. That is a statement about the model, and translating it
into a drug effect requires knowing how the drug maps onto the parameter — which
is exactly what the retrospective controls in `pharmacology.py` test, and which
the model currently fails for LOXL2.

**A consistency check is not an independent prediction.** The provisional
stiffness at which lesions begin to persist lands close to the stiffness at
which TGF-β is mechanically activated in the literature. That agreement is
reassuring, but both numbers came from the same body of work, so it is not
evidence the model got something right on its own.

**Defect counts are subject to the same noise floor as everywhere else.** Cell 2
prints the window radius needed for the cell size you chose. Below it, apparent
nematic order is dominated by counting noise.

## Relationship to the other notebook

`ipf_simulation_colab.ipynb` builds alveoli, epithelium, collapse and breathing,
and lets a focus emerge inside a collapsed alveolus. This one takes the focus as
given and dissects its stability. The alveolar model tells you *where* a lesion
forms; this one tells you *whether it can still go away*.